# 01 - Exploración de candidatos

**Objetivo:** bajar los precios de las acciones candidatas y confirmar con datos la selección: **10 objetivo** (las que el modelo predice y que salen en el menú de la app), **2 de contexto** (SPY y NVDA, que no se predicen y solo aportan señal de mercado) y un grupo de **reserva** por si alguna objetivo falla.

Este notebook es **exploración** (el documento del proyecto lo pide así: `notebooks/` no contiene código de producción). Lo que funcione aquí se pasará después a funciones en `src/`.

**Preguntas que responde:**
1. ¿Todas las acciones tienen historia suficiente desde la fecha de inicio?
2. ¿Hay datos faltantes o saltos raros que parezcan errores?
3. ¿Qué tan volátiles son y qué tan parecidas se mueven entre sí (correlación)?
4. ¿Qué porcentaje de días sube cada una? (es el *baseline*: el modelo que siempre responde "sube").

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf


def redondear(df, n=2):
    return df.round({c: n for c in df.select_dtypes("number").columns})

## 1. Parámetros

La selección sale de dos listas de tu plataforma de análisis. Está agrupada por tipo de negocio a propósito: si todas fueran de semiconductores, casi se moverían igual y el modelo vería la misma información varias veces. **No es una recomendación de inversión; es una muestra para el ejercicio.**

In [ ]:
INICIO = "2015-01-01"

OBJETIVO = {
    "Semiconductores": ["AMD"],
    "Software": ["ORCL"],
    "Servicios de informacion": ["IT"],
    "TI empresarial": ["IBM"],
    "Electronica de seguridad": ["OSIS"],
    "Defensa": ["LMT"],
    "Consumo basico (Mexico, ADR)": ["KOF"],
    "Salud (aseguradora)": ["HUM"],
    "Retail de ropa": ["ANF"],
    "Climatizacion (HVAC)": ["LII"],
}
CONTEXTO = {"Mercado (S&P 500)": ["SPY"], "Tecnologia grande": ["NVDA"]}
RESERVA = {
    "Semiconductores": ["QCOM", "CRUS", "INTC"],
    "TI": ["CTSH"],
    "Salud": ["GILD", "COR", "EXEL"],
    "Industria": ["AYI", "WSO"],
}

GRUPOS = {"objetivo": OBJETIVO, "contexto": CONTEXTO, "reserva": RESERVA}

rol_y_sector = {}
for rol, grupos in GRUPOS.items():
    for sector, lista in grupos.items():
        for t in lista:
            rol_y_sector[t] = (rol, sector)

tickers = list(rol_y_sector)
objetivo = [t for t, (rol, _) in rol_y_sector.items() if rol == "objetivo"]
contexto = [t for t, (rol, _) in rol_y_sector.items() if rol == "contexto"]
print(len(tickers), "tickers:", tickers)

## 2. Descarga (con caché)

`yf.download` devuelve columnas de dos niveles: primero el campo (`Close`, `Volume`...) y luego el ticker. Nos quedamos con `Close` ya **ajustado** (`auto_adjust=True`): corrige por splits y dividendos, que si no meterían saltos falsos en los retornos.

Se guarda una copia en `data/exploracion/` (Git la ignora) para no volver a descargar y para que el notebook 02 la reutilice. Si quieres datos frescos, borra esa carpeta.

In [ ]:
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CARPETA = RAIZ / "data" / "exploracion"
F_CIERRE, F_VOLUMEN = CARPETA / "close.csv", CARPETA / "volume.csv"

if F_CIERRE.exists() and F_VOLUMEN.exists():
    cierre = pd.read_csv(F_CIERRE, index_col=0, parse_dates=True)
    volumen = pd.read_csv(F_VOLUMEN, index_col=0, parse_dates=True)
else:
    datos = yf.download(tickers, start=INICIO, auto_adjust=True, progress=False)
    cierre, volumen = datos["Close"], datos["Volume"]
    CARPETA.mkdir(parents=True, exist_ok=True)
    cierre.to_csv(F_CIERRE)
    volumen.to_csv(F_VOLUMEN)

cierre = cierre[tickers]
volumen = volumen[tickers]
cierre.tail()

## 3. Revisión básica de la tabla

Cosas a revisar: tamaño, tipo del índice (debe ser fechas), orden, y desde cuándo hay datos para cada ticker.

In [ ]:
print("Forma (días x tickers):", cierre.shape)
print("Índice:", type(cierre.index).__name__)
print("¿Fechas ordenadas?:", cierre.index.is_monotonic_increasing)
print("¿Fechas repetidas?:", cierre.index.duplicated().any())
print("Rango:", cierre.index.min().date(), "->", cierre.index.max().date())

In [ ]:
resumen = pd.DataFrame(
    {
        "rol": pd.Series({t: v[0] for t, v in rol_y_sector.items()}),
        "sector": pd.Series({t: v[1] for t, v in rol_y_sector.items()}),
        "primer_dato": cierre.apply(lambda s: s.first_valid_index()),
        "dias_con_dato": cierre.notna().sum(),
        "dias_faltantes": cierre.isna().sum(),
    }
)
resumen["%_faltantes"] = resumen["dias_faltantes"] / len(cierre) * 100
redondear(resumen)

## 4. Retornos, volatilidad y días al alza

El **retorno diario** es el cambio porcentual del cierre. Se calcula con `fill_method=None` para **no rellenar** huecos: rellenar un día faltante con el precio anterior inventa un retorno de 0 %, y después un salto doble.

Trabajamos con retornos y no con precios porque el precio casi siempre sube con el tiempo y eso no generaliza entre acciones.

`% días al alza` es la proporción de días con retorno positivo. Un modelo que siempre dijera "sube" acertaría ese porcentaje: es el piso que nuestro modelo debe superar. Los retornos mínimo y máximo sirven para detectar errores de datos (un -60 % en un día sin motivo suele ser un dato malo, no una crisis).

In [ ]:
retornos = cierre.pct_change(fill_method=None).dropna(how="all")

resumen["retorno_medio_diario_%"] = retornos.mean() * 100
resumen["volatilidad_anual_%"] = retornos.std() * np.sqrt(252) * 100
resumen["dias_al_alza_%"] = (retornos > 0).sum() / retornos.notna().sum() * 100
resumen["ret_min_%"] = retornos.min() * 100
resumen["ret_max_%"] = retornos.max() * 100
redondear(resumen.drop(columns=["sector"]))

### Chequeos automáticos para las acciones objetivo

Reglas de la selección: historia desde la fecha de inicio (con unos días de tolerancia), menos de 2 % de días faltantes, y ningún retorno diario sospechoso (se marca cualquiera con más de 40 % en valor absoluto para revisarlo a mano; no se elimina solo).

In [ ]:
obj = resumen.loc[objetivo].copy()
limite_historia = pd.Timestamp(INICIO) + pd.Timedelta(days=10)
obj["historia_ok"] = obj["primer_dato"] <= limite_historia
obj["faltantes_ok"] = obj["%_faltantes"] < 2
obj["sin_saltos_raros"] = (obj["ret_min_%"] > -40) & (obj["ret_max_%"] < 40)
redondear(
    obj[
        [
            "primer_dato",
            "%_faltantes",
            "ret_min_%",
            "ret_max_%",
            "historia_ok",
            "faltantes_ok",
            "sin_saltos_raros",
        ]
    ]
)

In [ ]:
problemas = obj[~(obj["historia_ok"] & obj["faltantes_ok"] & obj["sin_saltos_raros"])]
if problemas.empty:
    print("Las 10 acciones objetivo pasan los tres chequeos.")
else:
    print("Revisar o reemplazar por una de reserva:", list(problemas.index))

## 5. Visualizaciones

El documento pide al menos tres gráficas distintas (distribución, serie de tiempo y comparación) con título, ejes y leyenda. Aquí van la serie de tiempo, la distribución y un mapa de calor de correlaciones; la comparación entre modelos viene en el notebook 02.

In [ ]:
base_100 = (
    cierre[objetivo + contexto]
    / cierre[objetivo + contexto].apply(lambda s: s.dropna().iloc[0])
    * 100
)

fig, ax = plt.subplots(figsize=(11, 5.5))
base_100.plot(ax=ax, linewidth=1)
ax.set_yscale("log")
ax.set_title("Evolución de objetivo y contexto (base 100 = primer dato, escala log)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Valor (base 100)")
ax.legend(ncol=4, fontsize=8)
plt.show()

In [ ]:
muestra = ["AMD", "KOF", "LMT", "SPY"]

fig, ax = plt.subplots(figsize=(9, 4.5))
for t in muestra:
    ax.hist(retornos[t].dropna(), bins=80, alpha=0.5, label=t, range=(-0.08, 0.08))
ax.set_title("Distribución de retornos diarios (rango recortado a +-8%)")
ax.set_xlabel("Retorno diario")
ax.set_ylabel("Número de días")
ax.legend()
plt.show()

In [ ]:
corr = retornos.corr()
orden = objetivo + contexto + [t for t in tickers if t not in objetivo + contexto]
corr = corr.loc[orden, orden]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.columns)
ax.set_title("Correlación de retornos diarios (objetivo, contexto y reserva)")
ax.set_xlabel("Ticker")
ax.set_ylabel("Ticker")
fig.colorbar(im, ax=ax, label="Correlación")
plt.show()

## 6. ¿Se parecen demasiado entre sí?

Para las 10 objetivo se listan las parejas más correlacionadas y la correlación de cada una con los dos tickers de contexto. Una correlación alta entre dos objetivo significa información repetida; una alta con SPY es normal (todas dependen del mercado) y solo indica que el contexto aporta poco por sí solo.

In [ ]:
c_obj = corr.loc[objetivo, objetivo]
pares = c_obj.where(np.triu(np.ones(c_obj.shape, dtype=bool), k=1)).stack()
pares = pares.sort_values(ascending=False).rename("correlacion")
print("Correlación promedio entre pares objetivo:", round(pares.mean(), 3))
pares.head(8).round(3)

In [ ]:
corr.loc[objetivo, contexto].round(3)

In [ ]:
parejas_altas = pares[pares > 0.6]
if parejas_altas.empty:
    print("Ninguna pareja objetivo supera 0.60 de correlación.")
else:
    print("Parejas con correlación > 0.60 (considerar reemplazar una):")
    print(parejas_altas.round(3))

## 7. Decisión

**Propuesta antes de ver datos:** objetivo = AMD, ORCL, IT, IBM, OSIS, LMT, KOF, HUM, ANF, LII; contexto = SPY y NVDA.

**Tickers objetivo finales:** AMD, ORCL, IT, IBM, OSIS, LMT, KOF, HUM, ANF, LII (sin cambios).

**Contexto:** SPY y NVDA. En el notebook 02 se probó agregar su retorno como característica y **empeoró** el AUC en los tres modelos probados, así que quedan fuera del modelo final; se pueden seguir usando como referencia visual.

**Razones:**
- **Historia:** las 10 tienen datos completos desde el 2 de enero de 2015 (2,960 días hasta el 9 de octubre de 2026) y **0 días faltantes**.
- **Correlación:** la correlación promedio entre pares objetivo es 0.23, y la pareja más parecida (IT e IBM) llega a 0.40, debajo del límite de 0.60 que nos pusimos. Es decir, aportan información distinta y no se repiten.
- **Volatilidad:** hay variedad, desde LMT con 23 % anual hasta AMD y ANF con 60 %.
- **Saltos raros:** el máximo diario de AMD (+52 % el 22 de abril de 2016) pasa el límite de 40 % del chequeo automático, pero es un movimiento real (resultados de la empresa), no un error de datos. Se conserva.

**Limitación a documentar:** las acciones se eligieron con una lista de hoy, así que hay sesgo de supervivencia (empresas que sobrevivieron hasta 2026).